# Maternal Health Risk Prediction

## ANN Model

#### Import Required Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import pickle
import os

print("Starting ANN script...")

Starting ANN script...


#### Load Data

In [2]:
data = pd.read_csv("../Dataset/Maternal Health Risk Data Set.csv")
print("Data loaded. Shape:", data.shape)

data = data.copy()

Data loaded. Shape: (1014, 7)


#### Data Information

In [3]:
print("="*60)
print("Dataset Information")
print("="*60)

display(data.info())

print()

display(data.describe())

print()

display(data.describe(include="object"))

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          1014 non-null   int64  
 1   SystolicBP   1014 non-null   int64  
 2   DiastolicBP  1014 non-null   int64  
 3   BS           1014 non-null   float64
 4   BodyTemp     1014 non-null   float64
 5   HeartRate    1014 non-null   int64  
 6   RiskLevel    1014 non-null   object 
dtypes: float64(2), int64(4), object(1)
memory usage: 55.6+ KB


None

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
count,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000
mean,29.871795,113.198225,76.460552,8.725986,98.665089,74.301775
std,13.474386,18.403913,13.885796,3.293532,1.371384,8.088702
min,10.000000,70.000000,49.000000,6.000000,98.000000,7.000000
25%,19.000000,100.000000,65.000000,6.900000,98.000000,70.000000
50%,26.000000,120.000000,80.000000,7.500000,98.000000,76.000000
75%,39.000000,120.000000,90.000000,8.000000,98.000000,80.000000
max,70.000000,160.000000,100.000000,19.000000,103.000000,90.000000


,RiskLevel
count,1014
unique,3
top,low risk
freq,406


#### Feature Engineering

In [4]:
if {"SystolicBP", "DiastolicBP"}.issubset(data.columns):
    data["PulsePressure"] = data["SystolicBP"] - data["DiastolicBP"]
    eps = 1e-3
    data["BPRatio"] = data["SystolicBP"] / (data["DiastolicBP"] + eps)

if {"BS", "HeartRate"}.issubset(data.columns):
    data["BS_HR_Interaction"] = data["BS"] * data["HeartRate"]

if {"Age", "SystolicBP"}.issubset(data.columns):
    data["Age_SystolicBP"] = data["Age"] * data["SystolicBP"]

TARGET_COL = "RiskLevel"

X = data.drop(columns=[TARGET_COL])
y = data[TARGET_COL]

print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts())

Features shape: (1014, 10)
Target distribution:
 RiskLevel
low risk     406
mid risk     336
high risk    272
Name: count, dtype: int64


#### Train/Test Split

In [5]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    print("Train set:", X_train.shape, "Test set:", X_test.shape)

Train set: (811, 10) Test set: (203, 10)


#### Pipeline (Scaler + ANN)

In [6]:
ann_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ann", MLPClassifier(max_iter=500, random_state=42))
])

#### Hyperparameter Grid

In [7]:
param_grid = {
    "ann__hidden_layer_sizes": [(64,), (128,), (64,32)],
    "ann__activation": ["relu", "tanh"],
    "ann__solver": ["adam"],     
    "ann__alpha": [0.0001, 0.001, 0.01]
}


cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#### Grid Search

In [8]:
    print("Running GridSearchCV...")

    grid_search = GridSearchCV(
        estimator=ann_pipe,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring="accuracy",
        n_jobs=-1,
        verbose=2
    )

    grid_search.fit(X_train, y_train)

    print("Best CV Accuracy:", grid_search.best_score_)
    print("Best Params:", grid_search.best_params_)

Running GridSearchCV...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best CV Accuracy: 0.7484662576687116
Best Params: {'ann__activation': 'relu', 'ann__alpha': 0.001, 'ann__hidden_layer_sizes': (64, 32), 'ann__solver': 'adam'}


#### Evaluate Best ANN Model

In [9]:
best_model = grid_search.best_estimator_

print("Training best model on full training data...")
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("\n===== FINAL TEST PERFORMANCE (BEST ANN MODEL) =====")
print("Test Accuracy:", acc)
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

print("Script finished.")

Training best model on full training data...

===== FINAL TEST PERFORMANCE (BEST ANN MODEL) =====
Test Accuracy: 0.6896551724137931

Confusion Matrix:
 [[48  3  4]
 [ 3 64 14]
 [ 2 37 28]]

Classification Report:
               precision    recall  f1-score   support

   high risk       0.91      0.87      0.89        55
    low risk       0.62      0.79      0.69        81
    mid risk       0.61      0.42      0.50        67

    accuracy                           0.69       203
   macro avg       0.71      0.69      0.69       203
weighted avg       0.69      0.69      0.68       203

Script finished.


#### Save ANN Model

In [10]:
print("="*70)
print("Saving ANN Model")
print("="*70)

os.makedirs("../Model", exist_ok=True)

MODEL_PATH = "../Model/ann.pkl"

with open(MODEL_PATH, "wb") as file:
    pickle.dump(best_model, file)

print("Model Saved Successfully")
print("Location :", MODEL_PATH)

Saving ANN Model
Model Saved Successfully
Location : ../Model/ann.pkl
